# Notebook 03 — Category Landscape Analysis

**Amazon Market Intelligence**  
**Questions answered:** Q1 (Where should I sell?), Q2 (Is my category growing or dying?), Q3 (How competitive is my space?)  
**Tool mode:** Category Scout  
**Gold tables:** `gold_subcategory_landscape` (248), `gold_main_category_landscape` (50), `gold_temporal_trends` (8,721)

---

## 0 — Setup

In [ ]:
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import os

# --- DuckDB connection ---
DB_PATH = os.path.join(os.getcwd(), 'data', 'amazon_intelligence.duckdb')
con = duckdb.connect(DB_PATH, read_only=True)

# --- Chart output directory ---
CHARTS_DIR = 'charts/03_category_landscape'
os.makedirs(CHARTS_DIR, exist_ok=True)

# --- Plotly defaults ---
TEMPLATE = 'plotly_white'
COLOR_SEQ = px.colors.qualitative.Set2

def save_chart(fig, name):
    """Save chart as interactive HTML and static PNG."""
    fig.write_html(f'{CHARTS_DIR}/{name}.html')
    try:
        fig.write_image(f'{CHARTS_DIR}/{name}.png', width=1200, height=700, scale=2)
    except Exception:
        pass  # kaleido not installed — skip PNG

print(f'Connected to: {DB_PATH}')
print(f'Charts → {CHARTS_DIR}/')

## 1 — Schema Discovery
Check what columns we have before building anything.

In [ ]:
for table in ['gold_subcategory_landscape', 'gold_main_category_landscape', 'gold_temporal_trends']:
    print(f'\n=== {table} ===')
    print(con.sql(f'DESCRIBE {table}').df().to_string())
    print(f'Rows: {con.sql(f"SELECT COUNT(*) FROM {table}").fetchone()[0]:,}')

## 2 — Load Gold Tables

In [ ]:
df_sub = con.sql('SELECT * FROM gold_subcategory_landscape').df()
df_main = con.sql('SELECT * FROM gold_main_category_landscape').df()
df_trends = con.sql('SELECT * FROM gold_temporal_trends').df()

print(f'Subcategory landscape: {df_sub.shape}')
print(f'Main category landscape: {df_main.shape}')
print(f'Temporal trends: {df_trends.shape}')

df_sub.head(3)

In [ ]:
df_sub.columns.tolist()

In [ ]:
df_main.columns.tolist()

In [ ]:
df_trends.columns.tolist()

---

## 3 — The Marketplace at a Glance

Before diving into categories, set the stage: how big is this marketplace and how alive is it?

In [ ]:
# Aggregate stats from subcategory landscape
# Adapt column names after schema discovery — these are best guesses from state doc findings
# Common expected columns: subcategory, product_count, total_revenue, avg_price, avg_rating,
#   active_pct (or active_rate), review_count, bestseller_count, etc.

# ---- ADAPT THESE after running cell 1 ----
# Uncomment and fix column names based on DESCRIBE output

# REVENUE_COL = 'total_revenue'     # or 'revenue', 'estimated_revenue'
# PRODUCTS_COL = 'product_count'    # or 'num_products', 'total_products'
# ACTIVE_COL = 'active_pct'         # or 'activity_rate', 'pct_active'
# SUBCAT_COL = 'subcategory'        # or 'sub_category', 'category_name'

print('⬆️  Run schema discovery (cell 1) first, then set column names above and re-run.')

### 3.1 — Top 20 Subcategories by Revenue

**Q1: Where should I sell?** — Follow the money.

In [ ]:
# ---- Replace column names after schema discovery ----
REVENUE_COL = 'total_revenue'   # FIX ME
PRODUCTS_COL = 'product_count'  # FIX ME
SUBCAT_COL = 'subcategory'      # FIX ME

top20_rev = df_sub.nlargest(20, REVENUE_COL)

fig = px.bar(
    top20_rev,
    x=REVENUE_COL,
    y=SUBCAT_COL,
    orientation='h',
    title='Top 20 Subcategories by Estimated Revenue',
    labels={REVENUE_COL: 'Estimated Revenue ($)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#2196F3']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
save_chart(fig, '01_top20_revenue')
fig.show()

### 3.2 — Revenue per Product (Demand Density)

Raw revenue is misleading — a category with $100M spread across 50K products is very different from $100M across 500.  
**Revenue per product** shows where demand concentrates.

In [ ]:
df_sub['revenue_per_product'] = df_sub[REVENUE_COL] / df_sub[PRODUCTS_COL].replace(0, np.nan)

top20_density = df_sub.nlargest(20, 'revenue_per_product')

fig = px.bar(
    top20_density,
    x='revenue_per_product',
    y=SUBCAT_COL,
    orientation='h',
    title='Top 20 Subcategories by Revenue per Product (Demand Density)',
    labels={'revenue_per_product': 'Revenue per Product ($)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
fig.update_traces(texttemplate='$%{x:,.0f}', textposition='outside')
save_chart(fig, '02_revenue_per_product')
fig.show()

### 3.3 — The Opportunity Quadrant

**X-axis:** Revenue per product (how much is being spent per listing)  
**Y-axis:** Activity rate (what % of listings actually sell)  
**Size:** Total product count (competition level)  

The sweet spot: **top-right, small bubble** = high demand per product, high activity, low competition.

In [ ]:
ACTIVE_COL = 'active_pct'  # FIX ME after schema discovery

fig = px.scatter(
    df_sub,
    x='revenue_per_product',
    y=ACTIVE_COL,
    size=PRODUCTS_COL,
    hover_name=SUBCAT_COL,
    title='Category Opportunity Quadrant — Where Demand Meets Activity',
    labels={
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate (%)',
        PRODUCTS_COL: 'Product Count'
    },
    template=TEMPLATE,
    color_discrete_sequence=['#4CAF50'],
    size_max=50
)

# Add quadrant lines at median
med_rev = df_sub['revenue_per_product'].median()
med_act = df_sub[ACTIVE_COL].median()
fig.add_hline(y=med_act, line_dash='dash', line_color='gray', opacity=0.5)
fig.add_vline(x=med_rev, line_dash='dash', line_color='gray', opacity=0.5)

# Label quadrants
fig.add_annotation(x=med_rev*3, y=med_act*1.3, text='🎯 Sweet Spot', showarrow=False,
                   font=dict(size=14, color='green'))
fig.add_annotation(x=med_rev*0.2, y=med_act*0.5, text='⚠️ Graveyard', showarrow=False,
                   font=dict(size=14, color='red'))

fig.update_layout(height=700)
save_chart(fig, '03_opportunity_quadrant')
fig.show()

### 3.4 — Activity Rate Distribution

**Q2 context:** How alive is the marketplace? What's the typical activity rate?

In [ ]:
fig = px.histogram(
    df_sub,
    x=ACTIVE_COL,
    nbins=30,
    title='Distribution of Activity Rates Across 248 Subcategories',
    labels={ACTIVE_COL: 'Activity Rate (% of products with sales > 0)'},
    template=TEMPLATE,
    color_discrete_sequence=['#9C27B0']
)
fig.add_vline(x=df_sub[ACTIVE_COL].median(), line_dash='dash', line_color='red',
              annotation_text=f'Median: {df_sub[ACTIVE_COL].median():.1f}%')
fig.update_layout(height=400)
save_chart(fig, '04_activity_distribution')
fig.show()

### 3.5 — Ghost Marketplace: Bottom 20 by Activity

Categories where most listings are dead weight.

In [ ]:
bottom20_activity = df_sub.nsmallest(20, ACTIVE_COL)

fig = px.bar(
    bottom20_activity,
    x=ACTIVE_COL,
    y=SUBCAT_COL,
    orientation='h',
    title='Bottom 20 Subcategories by Activity Rate — The Ghost Categories',
    labels={ACTIVE_COL: 'Activity Rate (%)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#F44336']
)
fig.update_layout(yaxis={'categoryorder': 'total descending'}, height=600)
fig.update_traces(texttemplate='%{x:.1f}%', textposition='outside')
save_chart(fig, '05_ghost_categories')
fig.show()

---

## 4 — Main Category Ecosystem (35M products)

Zoom out: the full Amazon landscape from McAuley metadata. This includes products with no sales data — the total catalog.

In [ ]:
df_main.head()

In [ ]:
# Main category column name — adapt after schema discovery
MAINCAT_COL = 'main_category'  # FIX ME
MAIN_PRODUCTS_COL = 'product_count'  # FIX ME  (may differ from subcategory table)

fig = px.bar(
    df_main.nlargest(25, MAIN_PRODUCTS_COL),
    x=MAIN_PRODUCTS_COL,
    y=MAINCAT_COL,
    orientation='h',
    title='Amazon Full Ecosystem — Top 25 Main Categories by Catalog Size (35M products)',
    labels={MAIN_PRODUCTS_COL: 'Total Products', MAINCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#00BCD4']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700)
fig.update_traces(texttemplate='%{x:,.0f}', textposition='outside')
save_chart(fig, '06_main_category_size')
fig.show()

### 4.1 — Catalog Size vs Brand Presence

Do bigger categories have stronger brand identity? Or is brand a niche advantage?

In [ ]:
# Adapt column names — gold_main_category_landscape likely has brand-related metrics
# Expected: brand_pct, avg_rating, avg_price, store_count, etc.
print(df_main.columns.tolist())
print()
df_main.describe()

---

## 5 — Temporal Trends (Q2: Growing or Dying?)

The temporal trends table has year-level review data by category. Review volume is a proxy for market activity.

In [ ]:
df_trends.head()

In [ ]:
# Adapt column names after schema discovery
YEAR_COL = 'year'  # FIX ME — could be 'review_year'
TREND_CAT_COL = 'source_category'  # FIX ME — could be 'category', 'main_category'
REVIEW_VOL_COL = 'review_count'  # FIX ME
AVG_RATING_COL = 'avg_rating'  # FIX ME

# Filter to meaningful year range (2018-2022, since 2023 is partial per Finding #36)
df_t = df_trends[df_trends[YEAR_COL].between(2018, 2022)].copy()

print(f'Year range in data: {df_trends[YEAR_COL].min()} — {df_trends[YEAR_COL].max()}')
print(f'Categories: {df_trends[TREND_CAT_COL].nunique()}')
print(f'Filtered rows (2018-2022): {len(df_t):,}')

### 5.1 — Overall Review Volume Trend

Is Amazon review activity growing or plateauing?

In [ ]:
yearly_total = df_t.groupby(YEAR_COL)[REVIEW_VOL_COL].sum().reset_index()

fig = px.bar(
    yearly_total,
    x=YEAR_COL,
    y=REVIEW_VOL_COL,
    title='Total Review Volume by Year (All Categories)',
    labels={REVIEW_VOL_COL: 'Reviews', YEAR_COL: 'Year'},
    template=TEMPLATE,
    color_discrete_sequence=['#3F51B5'],
    text_auto=True
)
fig.update_layout(height=400)
save_chart(fig, '07_review_volume_trend')
fig.show()

### 5.2 — Average Rating Decline

Finding #35: Ratings are declining 4.28 → 4.02 (2019-2022). Let's visualize the trajectory.

In [ ]:
# Weighted average rating per year (weight by review count)
yearly_rating = (
    df_t.groupby(YEAR_COL)
    .apply(lambda g: np.average(g[AVG_RATING_COL], weights=g[REVIEW_VOL_COL]))
    .reset_index(name='weighted_avg_rating')
)

fig = px.line(
    yearly_rating,
    x=YEAR_COL,
    y='weighted_avg_rating',
    title='Average Rating Decline Across Amazon (2018–2022)',
    labels={'weighted_avg_rating': 'Weighted Avg Rating', YEAR_COL: 'Year'},
    template=TEMPLATE,
    markers=True
)
fig.update_traces(line=dict(width=3, color='#E91E63'))
fig.update_yaxes(range=[3.8, 4.5])
fig.update_layout(height=400)
save_chart(fig, '08_rating_decline')
fig.show()

### 5.3 — Category Growth Heatmap

Which categories are gaining review momentum? Which are fading?  
Show year-over-year % change in review volume.

In [ ]:
# Pivot: rows = categories, columns = years, values = review count
pivot = df_t.pivot_table(
    index=TREND_CAT_COL,
    columns=YEAR_COL,
    values=REVIEW_VOL_COL,
    aggfunc='sum'
).fillna(0)

# YoY growth rate
growth = pivot.pct_change(axis=1) * 100
growth = growth.drop(columns=growth.columns[0])  # first year has no baseline

# Sort by most recent year growth
last_year = growth.columns[-1]
growth = growth.sort_values(last_year, ascending=True)

fig = px.imshow(
    growth,
    title='Category Growth Heatmap — YoY Review Volume Change (%)',
    labels=dict(x='Year', y='Category', color='YoY Change (%)'),
    color_continuous_scale='RdYlGn',
    color_continuous_midpoint=0,
    aspect='auto'
)
fig.update_layout(height=900)
save_chart(fig, '09_growth_heatmap')
fig.show()

### 5.4 — Top Growers vs Top Decliners

Rank categories by compound growth rate (2018→2022).

In [ ]:
# Compound growth: (end / start)^(1/years) - 1
start_year = pivot.columns[0]
end_year = pivot.columns[-1]
years = end_year - start_year

cagr = pd.DataFrame({
    TREND_CAT_COL: pivot.index,
    'start_vol': pivot[start_year].values,
    'end_vol': pivot[end_year].values
})
cagr = cagr[cagr['start_vol'] > 0]  # avoid division by zero
cagr['cagr_pct'] = ((cagr['end_vol'] / cagr['start_vol']) ** (1 / years) - 1) * 100

# Top 10 growers + bottom 10
top_growers = cagr.nlargest(10, 'cagr_pct')
top_decliners = cagr.nsmallest(10, 'cagr_pct')
extreme = pd.concat([top_growers, top_decliners]).sort_values('cagr_pct')

colors = ['#F44336' if x < 0 else '#4CAF50' for x in extreme['cagr_pct']]

fig = go.Figure(go.Bar(
    x=extreme['cagr_pct'],
    y=extreme[TREND_CAT_COL],
    orientation='h',
    marker_color=colors,
    text=[f'{x:+.1f}%' for x in extreme['cagr_pct']],
    textposition='outside'
))
fig.update_layout(
    title=f'Category Growth Champions & Decliners (CAGR {start_year}–{end_year})',
    xaxis_title='Compound Annual Growth Rate (%)',
    template=TEMPLATE,
    height=600
)
save_chart(fig, '10_growers_vs_decliners')
fig.show()

---

## 6 — Competition Intensity (Q3: How competitive is my space?)

Competition isn't just product count. It's the combination of:
- How many products exist (supply)
- How few are actually selling (activity rate)
- How concentrated revenue is (revenue per product)

Low activity + high product count = **oversaturated graveyard**.

In [ ]:
# Competition score: products / active_pct — high score = lots of products but few sell
df_sub['competition_score'] = df_sub[PRODUCTS_COL] / df_sub[ACTIVE_COL].replace(0, np.nan)

most_competitive = df_sub.nlargest(20, 'competition_score')

fig = px.bar(
    most_competitive,
    x='competition_score',
    y=SUBCAT_COL,
    orientation='h',
    title='Most Oversaturated Categories (High Product Count ÷ Low Activity)',
    labels={'competition_score': 'Competition Score (higher = harder)', SUBCAT_COL: ''},
    template=TEMPLATE,
    color_discrete_sequence=['#FF5722']
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=600)
save_chart(fig, '11_competition_intensity')
fig.show()

### 6.1 — Hidden Gems: High Demand, Low Competition

Categories with high revenue per product but relatively few competitors.

In [ ]:
# Opportunity score: revenue_per_product / product_count * activity_rate
# High revenue per product + few products + high activity = hidden gem
df_sub['opportunity_score'] = (
    df_sub['revenue_per_product'] * df_sub[ACTIVE_COL]
) / df_sub[PRODUCTS_COL].replace(0, np.nan)

gems = df_sub.nlargest(15, 'opportunity_score')

fig = px.scatter(
    gems,
    x=PRODUCTS_COL,
    y='revenue_per_product',
    size=ACTIVE_COL,
    hover_name=SUBCAT_COL,
    text=SUBCAT_COL,
    title='Hidden Gems — High Revenue per Product, Low Competition',
    labels={
        PRODUCTS_COL: 'Total Products (lower = less competition)',
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate'
    },
    template=TEMPLATE,
    color_discrete_sequence=['#FF9800'],
    size_max=40
)
fig.update_traces(textposition='top center', textfont_size=9)
fig.update_layout(height=600)
save_chart(fig, '12_hidden_gems')
fig.show()

---

## 7 — Category Typology

Classify every subcategory into one of 4 strategic types based on data:

| Type | Revenue/Product | Activity | What it means |
|------|----------------|----------|---------------|
| 🌟 Star | High | High | Money is here, things sell |
| 💀 Graveyard | Low | Low | Dead listings, nobody buys |
| 🏭 Volume Play | Low | High | Things sell but margins are thin |
| 💎 Niche Premium | High | Low | Few sell but those that do earn big |

In [ ]:
med_rev = df_sub['revenue_per_product'].median()
med_act = df_sub[ACTIVE_COL].median()

def classify(row):
    high_rev = row['revenue_per_product'] >= med_rev
    high_act = row[ACTIVE_COL] >= med_act
    if high_rev and high_act:
        return '🌟 Star'
    elif high_rev and not high_act:
        return '💎 Niche Premium'
    elif not high_rev and high_act:
        return '🏭 Volume Play'
    else:
        return '💀 Graveyard'

df_sub['category_type'] = df_sub.apply(classify, axis=1)

type_counts = df_sub['category_type'].value_counts()
print('Category Typology Distribution:')
for t, c in type_counts.items():
    print(f'  {t}: {c} subcategories')

# Scatter colored by type
fig = px.scatter(
    df_sub,
    x='revenue_per_product',
    y=ACTIVE_COL,
    color='category_type',
    hover_name=SUBCAT_COL,
    title='Category Typology — 248 Subcategories Classified',
    labels={
        'revenue_per_product': 'Revenue per Product ($)',
        ACTIVE_COL: 'Activity Rate (%)',
        'category_type': 'Type'
    },
    template=TEMPLATE,
    color_discrete_map={
        '🌟 Star': '#4CAF50',
        '💎 Niche Premium': '#FF9800',
        '🏭 Volume Play': '#2196F3',
        '💀 Graveyard': '#9E9E9E'
    }
)
fig.add_hline(y=med_act, line_dash='dash', line_color='gray', opacity=0.4)
fig.add_vline(x=med_rev, line_dash='dash', line_color='gray', opacity=0.4)
fig.update_layout(height=700)
save_chart(fig, '13_category_typology')
fig.show()

In [ ]:
# Revenue share by type
rev_by_type = df_sub.groupby('category_type')[REVENUE_COL].sum().reset_index()
rev_by_type['pct'] = (rev_by_type[REVENUE_COL] / rev_by_type[REVENUE_COL].sum() * 100).round(1)

fig = px.pie(
    rev_by_type,
    values=REVENUE_COL,
    names='category_type',
    title='Revenue Share by Category Type',
    template=TEMPLATE,
    color='category_type',
    color_discrete_map={
        '🌟 Star': '#4CAF50',
        '💎 Niche Premium': '#FF9800',
        '🏭 Volume Play': '#2196F3',
        '💀 Graveyard': '#9E9E9E'
    }
)
fig.update_traces(textinfo='label+percent', textfont_size=12)
fig.update_layout(height=450)
save_chart(fig, '14_revenue_by_type')
fig.show()

---

## 8 — Key Findings & Story Takeaways

Summarize what a Category Scout user would care about.

In [ ]:
# Programmatic summary
print('=' * 60)
print('CATEGORY LANDSCAPE — KEY FINDINGS')
print('=' * 60)

top1 = df_sub.nlargest(1, REVENUE_COL).iloc[0]
print(f'\n1. BIGGEST CATEGORY: {top1[SUBCAT_COL]}')
print(f'   Revenue: ${top1[REVENUE_COL]:,.0f} | Products: {top1[PRODUCTS_COL]:,.0f} | Activity: {top1[ACTIVE_COL]:.1f}%')

top_density = df_sub.nlargest(1, 'revenue_per_product').iloc[0]
print(f'\n2. HIGHEST DEMAND DENSITY: {top_density[SUBCAT_COL]}')
print(f'   Revenue/Product: ${top_density["revenue_per_product"]:,.0f}')

star_count = (df_sub['category_type'] == '🌟 Star').sum()
grave_count = (df_sub['category_type'] == '💀 Graveyard').sum()
print(f'\n3. TYPOLOGY: {star_count} Stars vs {grave_count} Graveyards (of 248 subcategories)')

star_rev_pct = rev_by_type[rev_by_type['category_type'] == '🌟 Star']['pct'].values
if len(star_rev_pct) > 0:
    print(f'   Stars capture {star_rev_pct[0]}% of total revenue')

print(f'\n4. RATING TREND: Declining across the marketplace')
if len(yearly_rating) >= 2:
    r_start = yearly_rating.iloc[0]['weighted_avg_rating']
    r_end = yearly_rating.iloc[-1]['weighted_avg_rating']
    print(f'   {r_start:.2f} → {r_end:.2f} ({r_start - r_end:+.2f} decline)')

print(f'\n5. TOP GROWERS: {top_growers[TREND_CAT_COL].iloc[0]} ({top_growers["cagr_pct"].iloc[0]:+.1f}% CAGR)')
print(f'   TOP DECLINERS: {top_decliners[TREND_CAT_COL].iloc[0]} ({top_decliners["cagr_pct"].iloc[0]:+.1f}% CAGR)')

print('\n' + '=' * 60)

In [ ]:
con.close()
print('Done. DuckDB connection closed.')
print(f'Charts saved to: {CHARTS_DIR}/')